Copyright **`(c)`** 2025 Giovanni Squillero `<giovanni.squillero@polito.it>`  
[`https://github.com/squillero/computational-intelligence`](https://github.com/squillero/computational-intelligence)  
Free under certain conditions — see the [`license`](https://github.com/squillero/computational-intelligence/blob/master/LICENSE.md) for details.  

In [1]:
from random import randint
from itertools import product
from collections import defaultdict
from pprint import pprint
from tqdm.auto import tqdm

from icecream import ic

In [10]:
NUM_ROWS = 5
NUM_COLS = 5
START = (NUM_ROWS - 1, 0)
GOAL = (0, NUM_COLS - 1)
MAP = [[(0, 0)] * NUM_COLS for _ in range(NUM_ROWS)]
for r, c in product(range(1, NUM_ROWS - 1), range(1, NUM_COLS - 1)):
    MAP[r][c] = (0, -1)
MAP

[[(0, 0), (0, 0), (0, 0), (0, 0), (0, 0)],
 [(0, 0), (0, -1), (0, -1), (0, -1), (0, 0)],
 [(0, 0), (0, -1), (0, -1), (0, -1), (0, 0)],
 [(0, 0), (0, -1), (0, -1), (0, -1), (0, 0)],
 [(0, 0), (0, 0), (0, 0), (0, 0), (0, 0)]]

In [ ]:
STATES = set(product(range(NUM_ROWS), range(NUM_COLS)))
ALL_ACTIONS = {(0, 0), (-1, 0), (0, 1), (1, 0), (-1, 0)}
ACTIONS = defaultdict(set)
for r, c in STATES:
    if r > 0:
        ACTIONS[(r, c)].add((-1, 0))
    if r < NUM_ROWS - 1:
        ACTIONS[(r, c)].add((1, 0))
    if c > 0:
        ACTIONS[(r, c)].add((0, -1))
    if c < NUM_COLS - 1:
        ACTIONS[(r, c)].add((0, 1))
ACTIONS[GOAL] = {(0, 0)}
REWARD = dict()
for s in STATES:
    for a in ACTIONS[s]:
        REWARD[(s, a)] = -1
REWARD[(GOAL, (0, 0))] = 0
len(REWARD)

79

In [12]:
NEXT_STATE = dict()
for s in STATES:
    for a in ACTIONS[s]:
        r = s[0] + a[0]
        c = s[1] + a[1]
        r += MAP[r][c][0]
        c += MAP[r][c][1]
        NEXT_STATE[s, a] = (r, c)

In [13]:
# STATE VALUE FUNCTION UNDER RANDOM POLICY
DISCOUNT = 0.9
V = defaultdict(float)
POLICY = ACTIONS

for i in tqdm(range(10_000)):
    V_old = V
    V = defaultdict(float)
    for s in STATES:
        for a in POLICY[s]:
            V[s] += (REWARD[s, a] + DISCOUNT * V_old[NEXT_STATE[s, a]]) / len(ACTIONS[s])

  0%|          | 0/10000 [00:00<?, ?it/s]

In [32]:
def create_greedy_policy(state_value):
    policy = dict()
    for s in STATES:
        max_ = max(state_value[s, a] for a in ACTIONS[s])
        policy[s] = { a for a in ACTIONS[s] if state_value[s, a] ==  max_}
    return policy

In [36]:
# STATE VALUE FUNCTION UNDER GREEDY POLICY

V = defaultdict(float)
policy = create_greedy_policy(V)

for steps in range(10):
    V = defaultdict(float)
    for i in tqdm(range(100)):
        V_old = V
        V = defaultdict(float)
        for s in STATES:
            for a in policy[s]:
                V[s] += (REWARD[s, a] + DISCOUNT * V_old[NEXT_STATE[s, a]]) / len(ACTIONS[s])
    policy = create_greedy_policy(V)


  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

In [37]:
POLICY

{(4, 0): {(-1, 0), (0, 1)},
 (3, 4): {(-1, 0), (0, -1), (1, 0)},
 (4, 3): {(-1, 0), (0, -1), (0, 1)},
 (3, 1): {(-1, 0), (0, -1), (0, 1), (1, 0)},
 (0, 2): {(0, -1), (0, 1), (1, 0)},
 (2, 2): {(-1, 0), (0, -1), (0, 1), (1, 0)},
 (1, 0): {(-1, 0), (0, 1), (1, 0)},
 (1, 3): {(-1, 0), (0, -1), (0, 1), (1, 0)},
 (4, 2): {(-1, 0), (0, -1), (0, 1)},
 (3, 0): {(-1, 0), (0, 1), (1, 0)},
 (3, 3): {(-1, 0), (0, -1), (0, 1), (1, 0)},
 (0, 1): {(0, -1), (0, 1), (1, 0)},
 (2, 4): {(-1, 0), (0, -1), (1, 0)},
 (1, 2): {(-1, 0), (0, -1), (0, 1), (1, 0)},
 (0, 4): {(0, 0)},
 (2, 1): {(-1, 0), (0, -1), (0, 1), (1, 0)},
 (3, 2): {(-1, 0), (0, -1), (0, 1), (1, 0)},
 (4, 1): {(-1, 0), (0, -1), (0, 1)},
 (4, 4): {(-1, 0), (0, -1)},
 (0, 0): {(0, 1), (1, 0)},
 (1, 1): {(-1, 0), (0, -1), (0, 1), (1, 0)},
 (0, 3): {(0, -1), (0, 1), (1, 0)},
 (2, 0): {(-1, 0), (0, 1), (1, 0)},
 (1, 4): {(-1, 0), (0, -1), (1, 0)},
 (2, 3): {(-1, 0), (0, -1), (0, 1), (1, 0)}}